# 10. 고위험군 예측 모델 학습 및 평가

**목적**  
제품·피부타입별 고위험군을 정의하고 V1/V2 특징과 여러 분류 모델을 비교합니다.

**입력**  
`data/processed의 최종 특징과 Gemini 감성분석 결과`

**출력**  
`reports/metrics/테스트_예측결과.csv`

> 저장된 데이터만 사용하며 크롤링이나 유료 API 호출은 발생하지 않습니다.


In [ ]:
from pathlib import Path

# Jupyter와 Colab 모두 저장소 루트에서 실행합니다.
def find_project_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "data").is_dir() and (path / "notebooks").is_dir():
            return path
    raise FileNotFoundError("저장소를 clone한 뒤 해당 폴더 안에서 실행하세요.")

PROJECT_ROOT = find_project_root()

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

for directory in [DATA_RAW_DIR, DATA_INTERIM_DIR, DATA_PROCESSED_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


## 실행 설정

입력 파일, 난수 시드와 Target 생성 기준을 설정합니다. 모델링은 저장된 특징과 감성분석 결과만 사용합니다.


In [ ]:
# 라이브러리 및 기본 설정
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

REVIEW_PATH = (
    DATA_PROCESSED_DIR
    / "Gemini_리뷰감성분석_배송제외_최종.csv"
)
V1_PATH = DATA_PROCESSED_DIR / "최종_V1_전체성분_V3V4V5.csv"
V2_PATH = DATA_PROCESSED_DIR / "최종_V2_선택성분_V3V4V5.csv"

ID_COLUMNS = ["product_id", "product_name", "피부타입", "formula_group"]
REVIEW_PRODUCT_COLUMN = "상품번호"
REVIEW_SKIN_COLUMN = "피부타입"
REVIEW_LABEL_COLUMN = "리뷰분류_1_0"

MIN_REVIEWS_PER_GROUP = 10
RISK_QUANTILE = 0.75
PRIOR_STRENGTH = 20


## 데이터 불러오기

Gemini 감성분석 결과와 V1·V2 최종 특징 데이터를 불러옵니다.


In [ ]:
# ============================================================
# 2. 데이터 불러오기
# ============================================================

def read_csv_with_fallback(path):
    """
    UTF-8, UTF-8-SIG, CP949 인코딩을 순서대로 시도한다.
    """
    encodings = ["utf-8-sig", "utf-8", "cp949"]
    errors = []

    for encoding in encodings:
        try:
            df = pd.read_csv(
                path,
                encoding=encoding,
                low_memory=False,
            )

            print(
                f"[로드 완료] {path.name} "
                f"| 인코딩={encoding} "
                f"| 크기={df.shape}"
            )

            return df

        except UnicodeDecodeError as error:
            errors.append(f"{encoding}: {error}")

    raise RuntimeError(
        f"{path.name} 파일의 인코딩을 확인하지 못했습니다.\n"
        + "\n".join(errors)
    )


# 리뷰 데이터
review_df = read_csv_with_fallback(REVIEW_PATH)

# V1 데이터
v1_df = read_csv_with_fallback(V1_PATH)

print(
    f"[로드 완료] {V1_PATH.name} "
    f"| 크기={v1_df.shape}"
)

# V2 데이터
v2_df = read_csv_with_fallback(V2_PATH)

## 입력 데이터 검증

각 파일의 필수 컬럼과 키 중복을 확인해 잘못된 병합을 사전에 방지합니다.


In [4]:
# ============================================================
# 3. 데이터 기본 검증
# ============================================================

def check_required_columns(df, required_columns, data_name):
    """
    데이터에 필수 컬럼이 존재하는지 확인한다.
    """
    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{data_name}에 필수 컬럼이 없습니다: "
            f"{missing_columns}"
        )

    print(f"[통과] {data_name} 필수 컬럼 확인")


# 리뷰 데이터 필수 컬럼
review_required_columns = [
    REVIEW_PRODUCT_COLUMN,
    REVIEW_SKIN_COLUMN,
    REVIEW_LABEL_COLUMN,
]

# V1, V2 필수 관리 컬럼
feature_required_columns = ID_COLUMNS

check_required_columns(
    review_df,
    review_required_columns,
    "리뷰 데이터",
)

check_required_columns(
    v1_df,
    feature_required_columns,
    "V1 데이터",
)

check_required_columns(
    v2_df,
    feature_required_columns,
    "V2 데이터",
)

[통과] 리뷰 데이터 필수 컬럼 확인
[통과] V1 데이터 필수 컬럼 확인
[통과] V2 데이터 필수 컬럼 확인


## 병합 키 정리

상품번호와 피부유형의 자료형·공백을 통일해 데이터 간 결합 기준을 맞춥니다.


In [5]:
# ============================================================
# 4. 병합 키 정리
# ============================================================

def clean_key(series):
    """
    병합에 사용할 키를 문자열로 통일하고
    앞뒤 공백과 비어 있는 값을 정리한다.
    """
    cleaned = series.astype("string").str.strip()

    cleaned = cleaned.replace(
        {
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
            "<NA>": pd.NA,
        }
    )

    return cleaned


# 리뷰 데이터 키 정리
review_df[REVIEW_PRODUCT_COLUMN] = clean_key(
    review_df[REVIEW_PRODUCT_COLUMN]
)

review_df[REVIEW_SKIN_COLUMN] = clean_key(
    review_df[REVIEW_SKIN_COLUMN]
)

# V1 키 정리
v1_df["product_id"] = clean_key(v1_df["product_id"])
v1_df["피부타입"] = clean_key(v1_df["피부타입"])
v1_df["formula_group"] = clean_key(v1_df["formula_group"])

# V2 키 정리
v2_df["product_id"] = clean_key(v2_df["product_id"])
v2_df["피부타입"] = clean_key(v2_df["피부타입"])
v2_df["formula_group"] = clean_key(v2_df["formula_group"])

print("병합 키 정리 완료")

병합 키 정리 완료


In [16]:
v1_df

,product_id,product_name,피부타입,formula_group,skin_건성,skin_민감성,skin_복합성,skin_약건성,skin_중성,skin_지성,...,v4_evidence_caution_score,v4_evidence_conflict_score,v5_theme_기타_강조문구_약함,v5_theme_남성_포맨,v5_theme_미백_잡티_톤업,v5_theme_보습_장벽_리페어,v5_theme_수분_수부지_아쿠아,v5_theme_진정_시카_민감,v5_theme_탄력_주름_안티에이징,v5_theme_피지_모공_트러블
0,A000000002848,바이오더마 세비엄 포어 리파이너,건성,f5f6b9ab23517e277ec36cdb5c3aeb67,1,0,0,0,0,0,...,0.000000,0.0,0,0,0,0,0,0,0,1
1,A000000002848,바이오더마 세비엄 포어 리파이너,약건성,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,0,1,0,0,...,0.000000,0.0,0,0,0,0,0,0,0,1
2,A000000002848,바이오더마 세비엄 포어 리파이너,지성,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,0,0,0,1,...,0.000000,0.0,0,0,0,0,0,0,0,1
3,A000000002848,바이오더마 세비엄 포어 리파이너,복합성,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,1,0,0,0,...,0.000000,0.0,0,0,0,0,0,0,0,1
4,A000000002848,바이오더마 세비엄 포어 리파이너,민감성,f5f6b9ab23517e277ec36cdb5c3aeb67,0,1,0,0,0,0,...,0.915311,0.0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1612,A000000260462,에스트라 아토베리어365 하이드로 수딩크림,지성,4f7d6fe03952a0b84b030e02d9352ed3,0,0,0,0,0,1,...,0.000000,0.0,0,0,0,1,1,1,0,0
1613,A000000260462,에스트라 아토베리어365 하이드로 수딩크림,복합성,4f7d6fe03952a0b84b030e02d9352ed3,0,0,1,0,0,0,...,0.000000,0.0,0,0,0,1,1,1,0,0
1614,A000000260462,에스트라 아토베리어365 하이드로 수딩크림,민감성,4f7d6fe03952a0b84b030e02d9352ed3,0,1,0,0,0,0,...,0.000000,0.0,0,0,0,1,1,1,0,0
1615,A000000260462,에스트라 아토베리어365 하이드로 수딩크림,트러블성,4f7d6fe03952a0b84b030e02d9352ed3,0,0,0,0,0,0,...,0.000000,0.0,0,0,0,1,1,1,0,0


## 리뷰 라벨 검증

긍정·부정 라벨과 제품·피부유형 정보가 모두 존재하는 리뷰만 Target 생성에 사용합니다.


In [6]:
# ============================================================
# 5. 리뷰 라벨 검증
# ============================================================

review_df[REVIEW_LABEL_COLUMN] = pd.to_numeric(
    review_df[REVIEW_LABEL_COLUMN],
    errors="coerce",
)

# 상품번호, 피부타입, 라벨이 모두 정상인 행만 선택
valid_review_mask = (
    review_df[REVIEW_PRODUCT_COLUMN].notna()
    & review_df[REVIEW_SKIN_COLUMN].notna()
    & review_df[REVIEW_LABEL_COLUMN].isin([0, 1])
)

invalid_review_count = int((~valid_review_mask).sum())

print(f"전체 리뷰 수: {len(review_df):,}")
print(f"유효하지 않은 리뷰 수: {invalid_review_count:,}")

# 유효 리뷰만 유지
review_clean = review_df.loc[valid_review_mask].copy()

review_clean[REVIEW_LABEL_COLUMN] = (
    review_clean[REVIEW_LABEL_COLUMN].astype(int)
)

print("\n[리뷰 라벨 분포]")
print(
    review_clean[REVIEW_LABEL_COLUMN]
    .value_counts()
    .sort_index()
)

print("\n0 = 부정, 1 = 긍정")

전체 리뷰 수: 47,487
유효하지 않은 리뷰 수: 0

[리뷰 라벨 분포]
리뷰분류_1_0
0     2780
1    44707
Name: count, dtype: int64

0 = 부정, 1 = 긍정


## 제품×피부유형별 리뷰 집계

과거 리뷰를 제품과 피부유형 단위로 묶어 전체·긍정·부정 리뷰 수와 부정 경험 비율을 계산합니다.


In [7]:
# ============================================================
# 6. 제품×피부타입별 리뷰 집계
# ============================================================

target_df = (
    review_clean
    .groupby(
        [
            REVIEW_PRODUCT_COLUMN,
            REVIEW_SKIN_COLUMN,
        ],
        as_index=False,
    )
    .agg(
        y_total_reviews=(
            REVIEW_LABEL_COLUMN,
            "count",
        ),
        y_positive_reviews=(
            REVIEW_LABEL_COLUMN,
            "sum",
        ),
    )
)

# 컬럼명 통일
target_df = target_df.rename(
    columns={
        REVIEW_PRODUCT_COLUMN: "product_id",
        REVIEW_SKIN_COLUMN: "피부타입",
    }
)

# 부정 리뷰 수
target_df["y_negative_reviews"] = (
    target_df["y_total_reviews"]
    - target_df["y_positive_reviews"]
)

# 부정 리뷰 비율
target_df["y_negative_rate"] = (
    target_df["y_negative_reviews"]
    / target_df["y_total_reviews"]
)

print(f"집계 전 리뷰 수: {len(review_clean):,}")
print(
    "제품×피부타입 조합 수: "
    f"{len(target_df):,}"
)

target_df.head()

집계 전 리뷰 수: 47,487
제품×피부타입 조합 수: 1,501


,product_id,피부타입,y_total_reviews,y_positive_reviews,y_negative_reviews,y_negative_rate
0,A000000002848,건성,56,52,4,0.071429
1,A000000002848,민감성,49,46,3,0.061224
2,A000000002848,복합성,53,52,1,0.018868
3,A000000002848,약건성,7,7,0,0.000000
4,A000000002848,중성,27,20,7,0.259259


## 최소 리뷰 수 적용

표본이 지나치게 적은 조합의 변동성을 줄이기 위해 리뷰가 10개 이상인 제품×피부유형 조합만 사용합니다.


In [10]:
# ============================================================
# 7. 최소 리뷰 수 필터링
# ============================================================

before_filter_count = len(target_df)

target_filtered = target_df.loc[
    target_df["y_total_reviews"]
    >= 10
].copy()

after_filter_count = len(target_filtered)

excluded_count = before_filter_count - after_filter_count

print("[최소 리뷰 수 필터링]")
print(f"최소 리뷰 수 기준: {MIN_REVIEWS_PER_GROUP}")
print(f"필터링 전 조합 수: {before_filter_count:,}")
print(f"필터링 후 조합 수: {after_filter_count:,}")
print(f"제외된 조합 수: {excluded_count:,}")

print("\n[피부타입별 남은 행 수]")
print(
    target_filtered["피부타입"]
    .value_counts()
    .sort_index()
)

[최소 리뷰 수 필터링]
최소 리뷰 수 기준: 10
필터링 전 조합 수: 1,501
필터링 후 조합 수: 1,152
제외된 조합 수: 349

[피부타입별 남은 행 수]
피부타입
건성      201
민감성     186
복합성     205
약건성      97
중성      113
지성      190
트러블성    160
Name: count, dtype: Int64


## High-Risk Target 생성

리뷰 수가 적은 조합의 극단적인 비율을 완화하도록 베이지안 평활화를 적용하고, 평활화 부정률 상위 25%를 High-Risk로 정의합니다.


In [ ]:
# ============================================================
# 8. high_risk 이진 타깃 생성
# ============================================================
# ============================================================
# 평활화된 부정률을 이용한 high_risk 생성
# (리뷰 수가 적은 조합에서 부정률이 극단적으로 튀는 현상을 방지하기 위해 전체 평균으로 보정합니다)
# ============================================================

# 전체 리뷰의 평균 부정률 계산 (전체 부정 리뷰 수 / 전체 리뷰 수)
global_negative_rate = (
    target_filtered["y_negative_reviews"].sum()
    / target_filtered["y_total_reviews"].sum()
)

# 원래 부정률 보존 (단순 비율: 부정 리뷰 수 / 총 리뷰 수)
target_filtered["y_negative_rate_raw"] = (
    target_filtered["y_negative_reviews"]
    / target_filtered["y_total_reviews"]
)

# 리뷰가 적은 조합은 전체 평균 쪽으로 보정 (베이지안 평활화 적용)
# 리뷰 수에 가상의 데이터(PRIOR_STRENGTH)를 추가하여 비율 보정
target_filtered["y_negative_rate_smoothed"] = (
    target_filtered["y_negative_reviews"]
    + PRIOR_STRENGTH * global_negative_rate
) / (
    target_filtered["y_total_reviews"]
    + PRIOR_STRENGTH
)

# 평활화된 부정률의 상위 25% 커트라인 탐색 (RISK_QUANTILE = 0.75)
risk_threshold = target_filtered[
    "y_negative_rate_smoothed"
].quantile(RISK_QUANTILE)

# 커트라인 이상인 경우 1(고위험군), 미만인 경우 0(일반군)으로 이진 변수 생성
target_filtered["high_risk"] = (
    target_filtered["y_negative_rate_smoothed"]
    >= risk_threshold
).astype(int)

# 확인을 위해 상품번호와 피부타입 기준으로 정렬
target_filtered = (
    target_filtered
    .sort_values(
        ["product_id", "피부타입"]
    )
    .reset_index(drop=True)
)

print("[평활화된 고위험군 생성 결과]")
print(f"전체 평균 부정률: {global_negative_rate:.4f}")
print(f"고위험군 기준: {risk_threshold:.4f}")

print("\n[Target 분포]")
print(
    target_filtered["high_risk"]
    .value_counts()
    .sort_index()
)


In [ ]:
print("\n[Target 비율]")
print(
    target_filtered["high_risk"]
    .value_counts(normalize=True)
    .sort_index()
)


In [13]:
target_filtered['y_negative_rate_smoothed'].quantile(0.75)

np.float64(0.07522143629926756)

In [14]:
target_filtered

,product_id,피부타입,y_total_reviews,y_positive_reviews,y_negative_reviews,y_negative_rate,y_negative_rate_raw,y_negative_rate_smoothed,high_risk
0,A000000002848,건성,56,52,4,0.071429,0.071429,0.067886,0
1,A000000002848,민감성,49,46,3,0.061224,0.061224,0.060280,0
2,A000000002848,복합성,53,52,1,0.018868,0.018868,0.029579,0
3,A000000002848,중성,27,20,7,0.259259,0.259259,0.173602,1
4,A000000002848,지성,56,50,6,0.107143,0.107143,0.094201,1
...,...,...,...,...,...,...,...,...,...
1147,A000000260348,중성,11,10,1,0.090909,0.090909,0.069655,0
1148,A000000260348,지성,43,38,5,0.116279,0.116279,0.097767,1
1149,A000000260348,트러블성,42,41,1,0.023810,0.023810,0.034827,0
1150,A000000260462,민감성,10,9,1,0.100000,0.100000,0.071977,0


In [17]:
# ============================================================
# 9. 생성된 Target 확인
# ============================================================

TARGET_COLUMNS = [
    "product_id",
    "피부타입",
    "y_total_reviews",
    "y_positive_reviews",
    "y_negative_reviews",
    "y_negative_rate_raw",
    "y_negative_rate_smoothed",
    "high_risk",
]

display(
    target_filtered[TARGET_COLUMNS].head(20)
)

,product_id,피부타입,y_total_reviews,y_positive_reviews,y_negative_reviews,y_negative_rate_raw,y_negative_rate_smoothed,high_risk
0,A000000002848,건성,56,52,4,0.071429,0.067886,0
1,A000000002848,민감성,49,46,3,0.061224,0.060280,0
2,A000000002848,복합성,53,52,1,0.018868,0.029579,0
3,A000000002848,중성,27,20,7,0.259259,0.173602,1
4,A000000002848,지성,56,50,6,0.107143,0.094201,1
5,A000000002848,트러블성,55,50,5,0.090909,0.082124,1
6,A000000010471,건성,22,22,0,0.000000,0.027602,0
7,A000000010471,복합성,41,36,5,0.121951,0.100972,1
8,A000000010471,지성,34,28,6,0.176471,0.132580,1
9,A000000010471,트러블성,14,12,2,0.142857,0.092921,1


## 특징과 Target 결합

V1·V2 특징 데이터에 제품번호와 피부유형을 기준으로 High-Risk Target을 연결합니다.


In [20]:
# ============================================================
# 11. V1·V2와 Target 병합
# ============================================================

MERGE_KEYS = ["product_id", "피부타입"]

TARGET_COLUMNS = [
    "product_id",
    "피부타입",
    "y_total_reviews",
    "y_positive_reviews",
    "y_negative_reviews",
    "y_negative_rate_raw",
    "y_negative_rate_smoothed",
    "high_risk",
]

# 필요한 Target 컬럼만 사용
target_for_merge = target_filtered[TARGET_COLUMNS].copy()

# V1 병합
v1_model_df = v1_df.merge(
    target_for_merge,
    on=MERGE_KEYS,
    how="inner",
    validate="one_to_one",
)

# V2 병합
v2_model_df = v2_df.merge(
    target_for_merge,
    on=MERGE_KEYS,
    how="inner",
    validate="one_to_one",
)

print("[병합 결과]")
print(f"V1 병합 전: {v1_df.shape}")
print(f"V1 병합 후: {v1_model_df.shape}")
print()
print(f"V2 병합 전: {v2_df.shape}")
print(f"V2 병합 후: {v2_model_df.shape}")

[병합 결과]
V1 병합 전: (1617, 1356)
V1 병합 후: (1124, 1362)

V2 병합 전: (1617, 312)
V2 병합 후: (1124, 318)


## 공통 평가 표본 구성

V1과 V2를 동일한 조건에서 비교하기 위해 두 특징 세트에 모두 존재하는 표본만 유지합니다.


In [ ]:
# ============================================================
# 12. V1·V2 공통 표본 생성
# ============================================================

v1_keys = v1_model_df[MERGE_KEYS].drop_duplicates()
v2_keys = v2_model_df[MERGE_KEYS].drop_duplicates()

common_keys = v1_keys.merge(
    v2_keys,
    on=MERGE_KEYS,
    how="inner",
    validate="one_to_one",
)

v1_model_df = (
    common_keys
    .merge(
        v1_model_df,
        on=MERGE_KEYS,
        how="left",
        validate="one_to_one",
    )
    .sort_values(MERGE_KEYS)
    .reset_index(drop=True)
)

v2_model_df = (
    common_keys
    .merge(
        v2_model_df,
        on=MERGE_KEYS,
        how="left",
        validate="one_to_one",
    )
    .sort_values(MERGE_KEYS)
    .reset_index(drop=True)
)

# V1과 V2의 표본 및 Y가 같은지 검증
assert len(v1_model_df) == len(v2_model_df)

assert v1_model_df["product_id"].equals(
    v2_model_df["product_id"]
)

assert v1_model_df["피부타입"].equals(
    v2_model_df["피부타입"]
)

assert v1_model_df["high_risk"].equals(
    v2_model_df["high_risk"]
)

print("[공통 표본]")
print(f"공통 행 수: {len(v1_model_df):,}")
print(f"공통 제품 수: {v1_model_df['product_id'].nunique():,}")
print(f"공통 처방 그룹 수: {v1_model_df['formula_group'].nunique():,}")
print()
print(v1_model_df["high_risk"].value_counts())
print()


In [23]:
print(v1_model_df["high_risk"].value_counts(normalize=True))


[공통 표본]
공통 행 수: 1,124
공통 제품 수: 206
공통 처방 그룹 수: 176

high_risk
0    837
1    287
Name: count, dtype: int64

high_risk
0    0.744662
1    0.255338
Name: proportion, dtype: float64


In [30]:
v2_keys

,product_id,피부타입
0,A000000002848,건성
1,A000000002848,민감성
2,A000000002848,복합성
3,A000000002848,중성
4,A000000002848,지성
...,...,...
1119,A000000260348,중성
1120,A000000260348,지성
1121,A000000260348,트러블성
1122,A000000260462,민감성


## 모델 입력 특징 선택

식별자와 리뷰 기반 Target 컬럼을 제외하고 피부유형 및 V1~V5에서 생성한 출시 전 특징만 선택합니다.


In [ ]:
# ============================================================
# 13. X 변수 선택
# ============================================================

ID_COLUMNS = [
    "product_id",
    "product_name",
    "피부타입",
    "formula_group",
]

Y_COLUMNS = [
    "y_total_reviews",
    "y_positive_reviews",
    "y_negative_reviews",
    "y_negative_rate_raw",
    "y_negative_rate_smoothed",
    "high_risk",
]


In [ ]:
def select_feature_columns(df, variant):
    """
    연구에서 정의한 변수만 선택한다.

    V1:
    skin_ + v1_ + v3 기능군 Count + v4_ + v5_theme_

    V2:
    skin_ + v2_ + v3 기능군 Count + v4_ + v5_theme_
    """

    feature_columns = []

    for column in df.columns:
        column = str(column)

        # 피부유형
        if column.startswith("skin_"):
            feature_columns.append(column)

        # V1 또는 V2 성분 변수
        elif variant == "V1" and column.startswith("v1_"):
            feature_columns.append(column)

        elif variant == "V2" and column.startswith("v2_"):
            feature_columns.append(column)

        # V3 기능군 Count 변수
        elif column == "v3_total_ingredient_count":
            feature_columns.append(column)

        elif column.startswith("v3_count_"):
            feature_columns.append(column)

        # V4 문헌 기반 변수
        elif column.startswith("v4_"):
            feature_columns.append(column)

        # V5 제품명 강조 문구
        elif column.startswith("v5_theme_"):
            feature_columns.append(column)

    # v3_ratio는 명시적으로 제외
    feature_columns = [
        column
        for column in feature_columns
        if not column.startswith("v3_ratio_")
    ]

    return feature_columns


v1_feature_columns = select_feature_columns(
    v1_model_df,
    "V1",
)


In [24]:
v2_feature_columns = select_feature_columns(
    v2_model_df,
    "V2",
)

print(f"V1 최초 X 변수 수: {len(v1_feature_columns):,}")
print(f"V2 최초 X 변수 수: {len(v2_feature_columns):,}")

print("\n[V1 변수 종류]")
print(pd.Series([
    "skin" if col.startswith("skin_")
    else "v1" if col.startswith("v1_")
    else "v3" if col.startswith("v3_")
    else "v4" if col.startswith("v4_")
    else "v5"
    for col in v1_feature_columns
]).value_counts())

print("\n[V2 변수 종류]")
print(pd.Series([
    "skin" if col.startswith("skin_")
    else "v2" if col.startswith("v2_")
    else "v3" if col.startswith("v3_")
    else "v4" if col.startswith("v4_")
    else "v5"
    for col in v2_feature_columns
]).value_counts())


V1 최초 X 변수 수: 1,352
V2 최초 X 변수 수: 308

[V1 변수 종류]
v1      1292
v4        32
v3        13
v5         8
skin       7
Name: count, dtype: int64

[V2 변수 종류]
v2      248
v4       32
v3       13
v5        8
skin      7
Name: count, dtype: int64


In [25]:
[col for col in v2_feature_columns if col.startswith('v3_')]

['v3_total_ingredient_count',
 'v3_count_humectant',
 'v3_count_emollient',
 'v3_count_occlusive',
 'v3_count_barrier',
 'v3_count_soothing',
 'v3_count_active',
 'v3_count_fragrance',
 'v3_count_acid_exfoliant',
 'v3_count_silicone',
 'v3_count_peptide',
 'v3_count_botanical_extract',
 'v3_count_oil_butter']

In [26]:
# X에 관리 컬럼과 Y가 들어가지 않았는지 확인
for column in ID_COLUMNS + Y_COLUMNS:
    assert column not in v1_feature_columns
    assert column not in v2_feature_columns

assert not any(
    column.startswith("v3_ratio_")
    for column in v1_feature_columns
)

assert not any(
    column.startswith("v3_ratio_")
    for column in v2_feature_columns
)

print("X 변수 누수 검사 통과")

X 변수 누수 검사 통과


In [28]:
v1_model_df

,product_id,피부타입,product_name,formula_group,skin_건성,skin_민감성,skin_복합성,skin_약건성,skin_중성,skin_지성,...,v5_theme_수분_수부지_아쿠아,v5_theme_진정_시카_민감,v5_theme_탄력_주름_안티에이징,v5_theme_피지_모공_트러블,y_total_reviews,y_positive_reviews,y_negative_reviews,y_negative_rate_raw,y_negative_rate_smoothed,high_risk
0,A000000002848,건성,바이오더마 세비엄 포어 리파이너,f5f6b9ab23517e277ec36cdb5c3aeb67,1,0,0,0,0,0,...,0,0,0,1,56,52,4,0.071429,0.067886,0
1,A000000002848,민감성,바이오더마 세비엄 포어 리파이너,f5f6b9ab23517e277ec36cdb5c3aeb67,0,1,0,0,0,0,...,0,0,0,1,49,46,3,0.061224,0.060280,0
2,A000000002848,복합성,바이오더마 세비엄 포어 리파이너,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,1,0,0,0,...,0,0,0,1,53,52,1,0.018868,0.029579,0
3,A000000002848,중성,바이오더마 세비엄 포어 리파이너,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,0,0,1,0,...,0,0,0,1,27,20,7,0.259259,0.173602,1
4,A000000002848,지성,바이오더마 세비엄 포어 리파이너,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,0,0,0,1,...,0,0,0,1,56,50,6,0.107143,0.094201,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1119,A000000260348,중성,리쥬덱스 더마 리페어링 크림,75084446fc859fb044eb70fcb65e2b69,0,0,0,0,1,0,...,0,0,1,0,11,10,1,0.090909,0.069655,0
1120,A000000260348,지성,리쥬덱스 더마 리페어링 크림,75084446fc859fb044eb70fcb65e2b69,0,0,0,0,0,1,...,0,0,1,0,43,38,5,0.116279,0.097767,1
1121,A000000260348,트러블성,리쥬덱스 더마 리페어링 크림,75084446fc859fb044eb70fcb65e2b69,0,0,0,0,0,0,...,0,0,1,0,42,41,1,0.023810,0.034827,0
1122,A000000260462,민감성,에스트라 아토베리어365 하이드로 수딩크림,4f7d6fe03952a0b84b030e02d9352ed3,0,1,0,0,0,0,...,1,1,0,0,10,9,1,0.100000,0.071977,0


In [27]:
# ============================================================
# 14. X, y, Group 준비
# ============================================================

X_v1_raw = (
    v1_model_df[v1_feature_columns]
    .apply(pd.to_numeric, errors="coerce")
)

X_v2_raw = (
    v2_model_df[v2_feature_columns]
    .apply(pd.to_numeric, errors="coerce")
)

y = v1_model_df["high_risk"].astype(int)

# 같은 처방 제품이 Train/Test 양쪽에 들어가지 않게 하는 그룹
groups = v1_model_df["formula_group"].astype(str)

# 결과 확인용 메타데이터
# 11번 셀에서 병합된 타깃 변수들을 모두 가져옵니다.
metadata = v1_model_df[
    [
        "product_id",
        "product_name",
        "피부타입",
        "formula_group",
        "y_total_reviews",
        "y_negative_rate_raw",
        "y_negative_rate_smoothed",
        "high_risk",
    ]
].copy()

print(f"X V1: {X_v1_raw.shape}")
print(f"X V2: {X_v2_raw.shape}")
print(f"y: {y.shape}")
print(f"Group 수: {groups.nunique():,}")

print("\n결측값")
print(f"V1: {X_v1_raw.isna().sum().sum():,}")
print(f"V2: {X_v2_raw.isna().sum().sum():,}")


X V1: (1124, 1352)
X V2: (1124, 308)
y: (1124,)
Group 수: 176

결측값
V1: 0
V2: 0


## Group 기반 Train/Test 분리

동일 처방이 Train과 Test에 동시에 포함되지 않도록 `formula_group` 단위로 분리하며, 두 세트의 양성 비율 차이도 함께 고려합니다.


In [ ]:
# ============================================================
# 15. Group 기반 Train/Test 분리
# ============================================================

from sklearn.model_selection import GroupShuffleSplit


def make_balanced_group_split(
    y,
    groups,
    test_size=0.2,
    n_candidates=200,
    random_state=42,
):
    """
    여러 GroupShuffleSplit 후보 중에서
    전체 고위험군 비율과 Test 고위험군 비율이
    가장 비슷한 분할을 선택한다.
    """

    splitter = GroupShuffleSplit(
        n_splits=n_candidates,
        test_size=test_size,
        random_state=random_state,
    )

    overall_rate = y.mean()

    best_train_index = None
    best_test_index = None
    best_score = float("inf")

    dummy_x = np.zeros((len(y), 1))

    for train_index, test_index in splitter.split(
        dummy_x,
        y,
        groups,
    ):
        train_rate = y.iloc[train_index].mean()
        test_rate = y.iloc[test_index].mean()

        score = (
            abs(train_rate - overall_rate)
            + abs(test_rate - overall_rate)
        )

        if score < best_score:
            best_score = score
            best_train_index = train_index
            best_test_index = test_index

    return best_train_index, best_test_index


In [31]:
train_index, test_index = make_balanced_group_split(
    y=y,
    groups=groups,
    test_size=0.2,
    n_candidates=300,
    random_state=RANDOM_STATE,
)

# V1
X_v1_train_raw = X_v1_raw.iloc[train_index].copy()
X_v1_test_raw = X_v1_raw.iloc[test_index].copy()

# V2
X_v2_train_raw = X_v2_raw.iloc[train_index].copy()
X_v2_test_raw = X_v2_raw.iloc[test_index].copy()

# 공통 Y
y_train = y.iloc[train_index].copy()
y_test = y.iloc[test_index].copy()

# Group
groups_train = groups.iloc[train_index].copy()
groups_test = groups.iloc[test_index].copy()

# 메타데이터
metadata_train = metadata.iloc[train_index].copy()
metadata_test = metadata.iloc[test_index].copy()

assert set(groups_train).isdisjoint(set(groups_test))

print("[Train/Test 분할]")
print(
    f"Train: {len(train_index):,}행, "
    f"{groups_train.nunique():,}개 처방 그룹"
)
print(
    f"Test: {len(test_index):,}행, "
    f"{groups_test.nunique():,}개 처방 그룹"
)

print(f"\n전체 고위험군 비율: {y.mean():.2%}")
print(f"Train 고위험군 비율: {y_train.mean():.2%}")
print(f"Test 고위험군 비율: {y_test.mean():.2%}")


[Train/Test 분할]
Train: 869행, 140개 처방 그룹
Test: 255행, 36개 처방 그룹

전체 고위험군 비율: 25.53%
Train 고위험군 비율: 25.55%
Test 고위험군 비율: 25.49%


## 특징 정리

Test 정보가 전처리에 영향을 주지 않도록 Train 데이터만 기준으로 상수·희귀·중복 특징을 제거합니다.


In [39]:
# ============================================================
# 16. 상수·희귀·중복 변수 제거
# ============================================================

def preprocess_feature_columns(
    X_train,
    X_test,
    min_nonzero=3,
):
    """
    Train 데이터만 사용하여:
    1. 상수 변수 제거
    2. 희귀 변수 제거
    3. 완전히 동일한 중복 변수 제거

    동일한 컬럼 선택을 Test에 적용한다.
    """

    X_train = X_train.copy()
    X_test = X_test.copy()

    original_count = X_train.shape[1]

    # 1. 상수 변수 제거
    nunique = X_train.nunique(dropna=False)

    constant_columns = nunique[
        nunique <= 1
    ].index.tolist()

    X_train = X_train.drop(
        columns=constant_columns
    )

    # 2. 희귀 변수 제거
    nonzero_counts = (X_train != 0).sum(axis=0)

    rare_columns = nonzero_counts[
        nonzero_counts < min_nonzero
    ].index.tolist()

    X_train = X_train.drop(
        columns=rare_columns
    )

    # 3. 동일한 값을 가진 중복 컬럼 제거
    duplicate_mask = X_train.T.duplicated()

    duplicate_columns = X_train.columns[
        duplicate_mask
    ].tolist()

    X_train = X_train.loc[
        :,
        ~duplicate_mask,
    ]

    selected_columns = X_train.columns.tolist()

    # Train에서 선택한 변수만 Test에 적용
    X_test = X_test[selected_columns].copy()

    report = {
        "original": original_count,
        "constant_removed": len(constant_columns),
        "rare_removed": len(rare_columns),
        "duplicate_removed": len(duplicate_columns),
        "final": len(selected_columns),
    }

    return X_train, X_test, selected_columns, report


In [40]:
X_v1_train, X_v1_test, v1_selected_columns, v1_report = (
    preprocess_feature_columns(
        X_v1_train_raw,
        X_v1_test_raw,
        min_nonzero=3,
    )
)

X_v2_train, X_v2_test, v2_selected_columns, v2_report = (
    preprocess_feature_columns(
        X_v2_train_raw,
        X_v2_test_raw,
        min_nonzero=3,
    )
)

print("[V1 전처리]")
print(v1_report)

print("\n[V2 전처리]")
print(v2_report)

[V1 전처리]
{'original': 1352, 'constant_removed': 205, 'rare_removed': 60, 'duplicate_removed': 405, 'final': 682}

[V2 전처리]
{'original': 308, 'constant_removed': 0, 'rare_removed': 0, 'duplicate_removed': 7, 'final': 301}


## Group 교차검증

Train 내부에서도 동일 처방이 Fold 사이에 섞이지 않도록 5-Fold Stratified Group Cross-Validation을 사용합니다.


In [41]:
# ============================================================
# 17. Stratified Group CV
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold

group_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# 각 Fold에서 Group 누수가 없는지 확인
for fold_number, (fit_index, valid_index) in enumerate(
    group_cv.split(
        X_v1_train,
        y_train,
        groups_train,
    ),
    start=1,
):
    fit_groups = set(groups_train.iloc[fit_index])
    valid_groups = set(groups_train.iloc[valid_index])

    assert fit_groups.isdisjoint(valid_groups)

    print(
        f"Fold {fold_number}: "
        f"Train={len(fit_index):,}, "
        f"Valid={len(valid_index):,}, "
        f"Valid 고위험군={y_train.iloc[valid_index].mean():.2%}"
    )

Fold 1: Train=697, Valid=172, Valid 고위험군=20.35%
Fold 2: Train=697, Valid=172, Valid 고위험군=25.00%
Fold 3: Train=696, Valid=173, Valid 고위험군=19.65%
Fold 4: Train=736, Valid=133, Valid 고위험군=34.59%
Fold 5: Train=650, Valid=219, Valid 고위험군=29.22%


In [44]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate

from xgboost import XGBClassifier
from catboost import CatBoostClassifier


In [45]:
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive
scale_pos_weight

np.float64(2.9144144144144146)

## 후보 모델 구성

Logistic Regression, Random Forest, XGBoost, CatBoost를 동일한 분할과 평가 지표로 비교합니다.


In [46]:
baseline_models = {
    "LogisticRegression": Pipeline(
        steps=[
            (
                "scaler",
                StandardScaler(with_mean=False),
            ),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=3000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        scale_pos_weight=scale_pos_weight,
      ),

    "CatBoost": CatBoostClassifier(
        iterations=300,
        depth=5,
        learning_rate=0.05,
        auto_class_weights="Balanced",
        verbose=False,
        allow_writing_files=False,
        random_seed=RANDOM_STATE,
    ),
}


In [47]:
scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

In [48]:
def evaluate_baseline_models(
    X_train,
    y_train,
    groups_train,
    variant,
):
    rows = []

    for model_name, model in baseline_models.items():
        print(f"실행 중: {variant} - {model_name}")

        scores = cross_validate(
            estimator=model,
            X=X_train,
            y=y_train,
            groups=groups_train,
            cv=group_cv,
            scoring=scoring,
            n_jobs=1,
            return_train_score=False,
        )

        row = {
            "variant": variant,
            "model": model_name,
        }

        for metric in scoring:
            values = scores[f"test_{metric}"]

            row[f"{metric}_mean"] = values.mean()


        rows.append(row)

    return pd.DataFrame(rows)


In [49]:
baseline_v1 = evaluate_baseline_models(
    X_v1_train,
    y_train,
    groups_train,
    "V1",
)

baseline_v2 = evaluate_baseline_models(
    X_v2_train,
    y_train,
    groups_train,
    "V2",
)

baseline_results = pd.concat(
    [baseline_v1, baseline_v2],
    ignore_index=True,
)

baseline_results = baseline_results.sort_values(
    ["pr_auc_mean", "f1_mean"],
    ascending=False,
)

display(baseline_results)

실행 중: V1 - LogisticRegression
실행 중: V1 - RandomForest
실행 중: V1 - XGBoost
실행 중: V1 - CatBoost
실행 중: V2 - LogisticRegression
실행 중: V2 - RandomForest
실행 중: V2 - XGBoost
실행 중: V2 - CatBoost


,variant,model,precision_mean,recall_mean,f1_mean,roc_auc_mean,pr_auc_mean
3,V1,CatBoost,0.308693,0.313492,0.307965,0.578979,0.334325
7,V2,CatBoost,0.355943,0.242210,0.287225,0.586481,0.333294
5,V2,RandomForest,0.620000,0.060317,0.107645,0.551359,0.331863
1,V1,RandomForest,0.537143,0.035845,0.066216,0.547381,0.326616
6,V2,XGBoost,0.296625,0.277496,0.285547,0.544009,0.294962
2,V1,XGBoost,0.280339,0.316314,0.295114,0.529278,0.290736
4,V2,LogisticRegression,0.257567,0.322347,0.280182,0.513268,0.285014
0,V1,LogisticRegression,0.249104,0.282745,0.253123,0.513520,0.274819


## 하이퍼파라미터 탐색

Train 교차검증의 PR-AUC를 기준으로 모델별 하이퍼파라미터를 탐색합니다. 독립 Test 세트는 모델 선정에 사용하지 않습니다.


In [50]:
# ============================================================
# 19. GridSearchCV
# ============================================================

from sklearn.model_selection import GridSearchCV

In [51]:
logistic_pipeline = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler(with_mean=False),
        ),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=5000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_grid = {
    "model__solver": ["liblinear"],
    "model__penalty": ["l1", "l2"],
    "model__C": [
        0.001,
        0.01,
        0.1,
        1,
        10,
    ],
}


In [52]:
grid_results = {}
roc_auc_results = {}  # ROC-AUC 성능을 기록하기 위한 딕셔너리 추가


In [53]:
group_cv

StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True)

In [54]:
for variant, X_train_variant in {
    "V1": X_v1_train,
    "V2": X_v2_train,
}.items():

    print(f"\n[{variant} LogisticRegression GridSearch]")

    search = GridSearchCV(
        estimator=logistic_pipeline,
        param_grid=logistic_grid,
        scoring={"pr_auc": "average_precision", "roc_auc": "roc_auc"},
        cv=group_cv,
        n_jobs=-1,
        refit="pr_auc",
        verbose=1,
    )

    search.fit(
        X_train_variant,
        y_train,
        groups=groups_train,
    )

    grid_results[
        f"{variant}_LogisticRegression"
    ] = search

    # PR-AUC가 가장 높은 모델의 ROC-AUC 결과 저장
    best_idx = search.best_index_
    best_roc_auc = search.cv_results_["mean_test_roc_auc"][best_idx]
    roc_auc_results[f"{variant}_LogisticRegression"] = best_roc_auc

    print("Best PR-AUC:", search.best_score_)
    print("Best ROC-AUC:", best_roc_auc)
    print("Best parameters:", search.best_params_)



[V1 LogisticRegression GridSearch]
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best PR-AUC: 0.318027856442965
Best ROC-AUC: 0.5748092522616373
Best parameters: {'model__C': 0.1, 'model__penalty': 'l1', 'model__solver': 'liblinear'}

[V2 LogisticRegression GridSearch]
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best PR-AUC: 0.32231419968489217
Best ROC-AUC: 0.5503889582064196
Best parameters: {'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'liblinear'}


In [55]:
rf_model = RandomForestClassifier(
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

rf_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 6, 12],
    "min_samples_split": [2, 10],
    "min_samples_leaf": [1, 5],
    "max_features": ["sqrt", 0.3],
}

In [56]:
for variant, X_train_variant in {
    "V1": X_v1_train,
    "V2": X_v2_train,
}.items():

    print(f"\n[{variant} RandomForest GridSearch]")

    search = GridSearchCV(
        estimator=rf_model,
        param_grid=rf_grid,
        scoring={"pr_auc": "average_precision", "roc_auc": "roc_auc"},
        cv=group_cv,
        n_jobs=-1,
        refit="pr_auc",
        verbose=1,
    )

    search.fit(
        X_train_variant,
        y_train,
        groups=groups_train,
    )

    grid_results[
        f"{variant}_RandomForest"
    ] = search

    # PR-AUC가 가장 높은 모델의 ROC-AUC 결과 저장
    best_idx = search.best_index_
    best_roc_auc = search.cv_results_["mean_test_roc_auc"][best_idx]
    roc_auc_results[f"{variant}_RandomForest"] = best_roc_auc

    print("Best PR-AUC:", search.best_score_)
    print("Best ROC-AUC:", best_roc_auc)
    print("Best parameters:", search.best_params_)



[V1 RandomForest GridSearch]
Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best PR-AUC: 0.3311065022492323
Best ROC-AUC: 0.5507091338042397
Best parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}

[V2 RandomForest GridSearch]
Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best PR-AUC: 0.3523984504799521
Best ROC-AUC: 0.5581801412780246
Best parameters: {'max_depth': 6, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 200}


In [58]:
import optuna

from sklearn.metrics import average_precision_score

In [ ]:
from sklearn.metrics import roc_auc_score


In [59]:
def optuna_xgboost(
    X_train,
    y_train,
    groups_train,
    n_trials=30,
):
    # 1. 최적화할 목적 함수(Objective Function) 정의
    def objective(trial):
        # Optuna가 탐색할 하이퍼파라미터 공간 정의
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 150, 600, step=50),  # 트리의 개수
            "max_depth": trial.suggest_int("max_depth", 2, 7),                     # 트리의 최대 깊이
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True), # 학습률 (로그 스케일 탐색)
            "min_child_weight": trial.suggest_float("min_child_weight", 1, 15),    # 자식 노드에 필요한 최소 가중치 합
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),               # 각 트리를 학습할 때 사용할 데이터 샘플 비율
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0), # 각 트리를 학습할 때 사용할 특성(feature) 샘플 비율
            "gamma": trial.suggest_float("gamma", 0, 5),                           # 리프 노드의 추가 분할을 결정할 최소 손실 감소량
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 10, log=True),     # L1 정규화 (Lasso)
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 20, log=True),    # L2 정규화 (Ridge)
        }

        pr_auc_scores = []
        roc_auc_scores = []

        # 2. 교차 검증 (Cross Validation) 수행
        for fit_index, valid_index in group_cv.split(
            X_train,
            y_train,
            groups_train,
        ):
            X_fit = X_train.iloc[fit_index]
            X_valid = X_train.iloc[valid_index]

            y_fit = y_train.iloc[fit_index]
            y_valid = y_train.iloc[valid_index]

            # 클래스 불균형 해결을 위한 scale_pos_weight 계산
            negative = (y_fit == 0).sum()
            positive = (y_fit == 1).sum()
            scale_pos_weight = negative / positive

            # 3. XGBoost 모델 생성 및 설정
            model = XGBClassifier(
                **params,
                objective="binary:logistic", # 이진 분류 문제
                eval_metric="logloss",     # 평가 지표
                tree_method="hist",        # 히스토그램 기반 트리 구축 (속도 향상)
                scale_pos_weight=scale_pos_weight, # 클래스 가중치 적용
                n_jobs=-1,                 # 모든 코어 사용
                random_state=RANDOM_STATE, # 재현성을 위한 난수 고정
            )

            # 4. 모델 학습
            model.fit(
                X_fit,
                y_fit,
                verbose=False,
            )

            # 5. 검증 데이터에 대한 예측 확률 계산 (양성 클래스 확률)
            probabilities = model.predict_proba(
                X_valid
            )[:, 1]

            # 6. PR-AUC 및 ROC-AUC 점수 계산
            pr_auc = average_precision_score(
                y_valid,
                probabilities,
            )
            roc_auc = roc_auc_score(
                y_valid,
                probabilities,
            )

            pr_auc_scores.append(pr_auc)
            roc_auc_scores.append(roc_auc)

        mean_pr_auc = float(np.mean(pr_auc_scores))
        mean_roc_auc = float(np.mean(roc_auc_scores))

        # Trial 속성에 ROC-AUC 기록
        trial.set_user_attr("roc_auc", mean_roc_auc)

        # 교차 검증 폴드들의 평균 PR-AUC 점수를 반환 (이 값을 최대화하는 것이 목표)
        return mean_pr_auc

    # Optuna Study 객체 생성 (방향: 최대화)
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler( # TPE(Tree-structured Parzen Estimator) 샘플링 알고리즘 사용
            seed=RANDOM_STATE
        ),
    )

    # 최적화 수행
    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=True,
    )

    return study


In [60]:
xgb_study_v1 = optuna_xgboost(
    X_v1_train,
    y_train,
    groups_train,
    n_trials=10,
)

print("V1 Best PR-AUC:", xgb_study_v1.best_value)
print(xgb_study_v1.best_params)

[I 2026-08-02 15:57:03,826] A new study created in memory with name: no-name-9e5fe9a0-650b-469d-b7ec-b785e0628958


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-08-02 15:57:20,063] Trial 0 finished with value: 0.30764790102476175 and parameters: {'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.08960785365368121, 'min_child_weight': 9.381218778758512, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.40919616423534183, 'gamma': 0.2904180608409973, 'reg_alpha': 1.574189004745663, 'reg_lambda': 2.416482602989751}. Best is trial 0 with value: 0.30764790102476175.
[I 2026-08-02 15:57:39,094] Trial 1 finished with value: 0.31927264507250025 and parameters: {'n_estimators': 500, 'max_depth': 2, 'learning_rate': 0.18276027831785724, 'min_child_weight': 12.654196971205904, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.42727747704497043, 'gamma': 0.9170225492671691, 'reg_alpha': 0.0006690421166498799, 'reg_lambda': 1.6124278458562613}. Best is trial 1 with value: 0.31927264507250025.
[I 2026-08-02 15:57:53,408] Trial 2 finished with value: 0.295699536860688 and parameters: {'n_estimators': 350, 'max_depth': 3, 'learning_rat

In [61]:
xgb_study_v2 = optuna_xgboost(
    X_v2_train,
    y_train,
    groups_train,
    n_trials=10,
)

print("V2 Best PR-AUC:", xgb_study_v2.best_value)
print(xgb_study_v2.best_params)

[I 2026-08-02 15:59:31,614] A new study created in memory with name: no-name-e9eacb9e-84a0-4a9b-a267-9e83d657acc0


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-08-02 15:59:36,413] Trial 0 finished with value: 0.33627225431836205 and parameters: {'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.08960785365368121, 'min_child_weight': 9.381218778758512, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.40919616423534183, 'gamma': 0.2904180608409973, 'reg_alpha': 1.574189004745663, 'reg_lambda': 2.416482602989751}. Best is trial 0 with value: 0.33627225431836205.
[I 2026-08-02 15:59:44,882] Trial 1 finished with value: 0.3261443685767932 and parameters: {'n_estimators': 500, 'max_depth': 2, 'learning_rate': 0.18276027831785724, 'min_child_weight': 12.654196971205904, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.42727747704497043, 'gamma': 0.9170225492671691, 'reg_alpha': 0.0006690421166498799, 'reg_lambda': 1.6124278458562613}. Best is trial 0 with value: 0.33627225431836205.
[I 2026-08-02 15:59:49,729] Trial 2 finished with value: 0.3104441328881739 and parameters: {'n_estimators': 350, 'max_depth': 3, 'learning_rat

In [ ]:
def optuna_catboost(
    X_train,
    y_train,
    groups_train,
    n_trials=30,
):
    # 1. 목적 함수(Objective Function) 정의: Optuna가 최적화할 대상
    def objective(trial):
        # 탐색할 하이퍼파라미터 공간 정의
        params = {
            "iterations": trial.suggest_int("iterations", 100, 500, step=50), # 트리 개수
            "depth": trial.suggest_int("depth", 3, 8),                        # 트리의 깊이
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True), # 학습률
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10, log=True),         # L2 정규화
            "random_strength": trial.suggest_float("random_strength", 0.1, 10, log=True), # 무작위성 부여 (과적합 방지)
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),  # 배깅 온도 (샘플 가중치)
        }

        pr_auc_scores = []
        roc_auc_scores = []

        # 2. 교차 검증 (Cross Validation) 수행 (그룹 누수 방지)
        for fit_index, valid_index in group_cv.split(
            X_train,
            y_train,
            groups_train,
        ):
            X_fit = X_train.iloc[fit_index]
            X_valid = X_train.iloc[valid_index]

            y_fit = y_train.iloc[fit_index]
            y_valid = y_train.iloc[valid_index]

            # 3. CatBoost 모델 생성
            model = CatBoostClassifier(
                **params,
                auto_class_weights="Balanced", # 클래스 불균형 자동 처리
                verbose=False,
                allow_writing_files=False,
                random_seed=RANDOM_STATE,
            )

            # 4. 모델 학습
            model.fit(X_fit, y_fit)

            # 5. 검증 데이터로 예측 확률 계산 (양성 클래스 확률)
            probabilities = model.predict_proba(X_valid)[:, 1]

            # 6. 평가 지표 계산 (PR-AUC, ROC-AUC)
            pr_auc = average_precision_score(y_valid, probabilities)
            roc_auc = roc_auc_score(y_valid, probabilities)

            pr_auc_scores.append(pr_auc)
            roc_auc_scores.append(roc_auc)

        # CV 폴드의 평균 점수 계산
        mean_pr_auc = float(np.mean(pr_auc_scores))
        mean_roc_auc = float(np.mean(roc_auc_scores))

        # 부가적인 정보(ROC-AUC)를 trial에 저장하여 나중에 확인할 수 있게 함
        trial.set_user_attr("roc_auc", mean_roc_auc)

        # 목적 함수는 최적화하려는 주 지표인 PR-AUC의 평균값을 반환
        return mean_pr_auc

    # 7. Optuna Study 객체 생성 (방향: 최대화, 샘플러: TPE)
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )

    # 8. 하이퍼파라미터 최적화 실행
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    return study


In [62]:
print("CatBoost Optuna function ready.")


CatBoost Optuna function ready.


In [63]:
cb_study_v1 = optuna_catboost(
    X_v1_train,
    y_train,
    groups_train,
    n_trials=10,
)

print("V1 CatBoost Best PR-AUC:", cb_study_v1.best_value)
print(cb_study_v1.best_params)

[I 2026-08-02 16:00:53,015] A new study created in memory with name: no-name-cbe920ee-70df-4ae8-a99a-c2eecf7892e2


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-08-02 16:01:02,627] Trial 0 finished with value: 0.30151849959262045 and parameters: {'iterations': 250, 'depth': 8, 'learning_rate': 0.08960785365368121, 'l2_leaf_reg': 3.968793330444372, 'random_strength': 0.20513382630874505, 'bagging_temperature': 0.15599452033620265}. Best is trial 0 with value: 0.30151849959262045.
[I 2026-08-02 16:01:06,799] Trial 1 finished with value: 0.30909982959781396 and parameters: {'iterations': 100, 'depth': 8, 'learning_rate': 0.06054365855469246, 'l2_leaf_reg': 5.105903209394756, 'random_strength': 0.10994335574766201, 'bagging_temperature': 0.9699098521619943}. Best is trial 1 with value: 0.30909982959781396.
[I 2026-08-02 16:01:11,296] Trial 2 finished with value: 0.31323603180163107 and parameters: {'iterations': 450, 'depth': 4, 'learning_rate': 0.017240892195821537, 'l2_leaf_reg': 1.5254729458052607, 'random_strength': 0.4059611610484306, 'bagging_temperature': 0.5247564316322378}. Best is trial 2 with value: 0.31323603180163107.
[I 2026-

In [64]:
cb_study_v2 = optuna_catboost(
    X_v2_train,
    y_train,
    groups_train,
    n_trials=10,
)

print("V2 CatBoost Best PR-AUC:", cb_study_v2.best_value)
print(cb_study_v2.best_params)

[I 2026-08-02 16:01:48,084] A new study created in memory with name: no-name-acdef310-ddda-4f72-bfeb-361a5a4b7b73


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-08-02 16:01:58,101] Trial 0 finished with value: 0.3133029929478409 and parameters: {'iterations': 250, 'depth': 8, 'learning_rate': 0.08960785365368121, 'l2_leaf_reg': 3.968793330444372, 'random_strength': 0.20513382630874505, 'bagging_temperature': 0.15599452033620265}. Best is trial 0 with value: 0.3133029929478409.
[I 2026-08-02 16:02:02,343] Trial 1 finished with value: 0.3203344693082113 and parameters: {'iterations': 100, 'depth': 8, 'learning_rate': 0.06054365855469246, 'l2_leaf_reg': 5.105903209394756, 'random_strength': 0.10994335574766201, 'bagging_temperature': 0.9699098521619943}. Best is trial 1 with value: 0.3203344693082113.
[I 2026-08-02 16:02:05,776] Trial 2 finished with value: 0.32630061867757043 and parameters: {'iterations': 450, 'depth': 4, 'learning_rate': 0.017240892195821537, 'l2_leaf_reg': 1.5254729458052607, 'random_strength': 0.4059611610484306, 'bagging_temperature': 0.5247564316322378}. Best is trial 2 with value: 0.32630061867757043.
[I 2026-08-0

In [66]:
grid_results

{'V1_LogisticRegression': GridSearchCV(cv=StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True),
              estimator=Pipeline(steps=[('scaler',
                                         StandardScaler(with_mean=False)),
                                        ('model',
                                         LogisticRegression(class_weight='balanced',
                                                            max_iter=5000,
                                                            random_state=42))]),
              n_jobs=-1,
              param_grid={'model__C': [0.001, 0.01, 0.1, 1, 10],
                          'model__penalty': ['l1', 'l2'],
                          'model__solver': ['liblinear']},
              refit='pr_auc',
              scoring={'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'},
              verbose=1),
 'V2_LogisticRegression': GridSearchCV(cv=StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True),
              estimator=Pipel

In [67]:
roc_auc_results

{'V1_LogisticRegression': np.float64(0.5748092522616373),
 'V2_LogisticRegression': np.float64(0.5503889582064196),
 'V1_RandomForest': np.float64(0.5507091338042397),
 'V2_RandomForest': np.float64(0.5581801412780246)}

## 최종 후보 비교

V1·V2와 네 분류 모델의 교차검증 결과를 한 표로 비교해 최종 후보를 선택합니다.


In [65]:
candidate_rows = []

for label, search in grid_results.items():
    candidate_rows.append({
        "label": label,
        "method": "GridSearchCV",
        "cv_pr_auc": search.best_score_,
        "cv_roc_auc": roc_auc_results[label],
        "parameters": search.best_params_,
    })

candidate_rows.append({
    "label": "V1_XGBoost",
    "method": "Optuna",
    "cv_pr_auc": xgb_study_v1.best_value,
    "cv_roc_auc": xgb_study_v1.best_trial.user_attrs.get("roc_auc"),
    "parameters": xgb_study_v1.best_params,
})

candidate_rows.append({
    "label": "V2_XGBoost",
    "method": "Optuna",
    "cv_pr_auc": xgb_study_v2.best_value,
    "cv_roc_auc": xgb_study_v2.best_trial.user_attrs.get("roc_auc"),
    "parameters": xgb_study_v2.best_params,
})

candidate_rows.append({
    "label": "V1_CatBoost",
    "method": "Optuna",
    "cv_pr_auc": cb_study_v1.best_value,
    "cv_roc_auc": cb_study_v1.best_trial.user_attrs.get("roc_auc"),
    "parameters": cb_study_v1.best_params,
})

candidate_rows.append({
    "label": "V2_CatBoost",
    "method": "Optuna",
    "cv_pr_auc": cb_study_v2.best_value,
    "cv_roc_auc": cb_study_v2.best_trial.user_attrs.get("roc_auc"),
    "parameters": cb_study_v2.best_params,
})

candidate_results = (
    pd.DataFrame(candidate_rows)
    .sort_values(
        "cv_pr_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(candidate_results)


,label,method,cv_pr_auc,cv_roc_auc,parameters
0,V2_RandomForest,GridSearchCV,0.352398,0.558180,"{'max_depth': 6, 'max_features': 'sqrt', 'min_..."
1,V1_CatBoost,Optuna,0.337570,0.551024,"{'iterations': 350, 'depth': 4, 'learning_rate..."
2,V2_XGBoost,Optuna,0.336272,0.565035,"{'n_estimators': 300, 'max_depth': 7, 'learnin..."
3,V1_RandomForest,GridSearchCV,0.331107,0.550709,"{'max_depth': None, 'max_features': 'sqrt', 'm..."
4,V2_CatBoost,Optuna,0.329328,0.575748,"{'iterations': 350, 'depth': 4, 'learning_rate..."
5,V2_LogisticRegression,GridSearchCV,0.322314,0.550389,"{'model__C': 0.01, 'model__penalty': 'l2', 'mo..."
6,V1_XGBoost,Optuna,0.322108,0.540292,"{'n_estimators': 400, 'max_depth': 7, 'learnin..."
7,V1_LogisticRegression,GridSearchCV,0.318028,0.574809,"{'model__C': 0.1, 'model__penalty': 'l1', 'mod..."


## 임계값 선택과 독립 Test 평가

선택된 모델의 Train OOF 예측으로 분류 임계값을 결정한 후, 한 번도 사용하지 않은 Test 세트에서 최종 성능을 평가합니다.


In [ ]:
# ============================================================
# 최종 후보 모델 선택 및 Test 평가
# ============================================================

from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

# ------------------------------------------------------------
# 1. Train CV 결과가 가장 좋은 후보 확인
# ------------------------------------------------------------

best_model_info = candidate_results.iloc[0]

best_label = best_model_info["label"]
best_params = best_model_info["parameters"]

print("선택된 최종 후보:", best_label)
print("CV PR-AUC:", best_model_info["cv_pr_auc"])
print("최적 파라미터:", best_params)


# ------------------------------------------------------------
# 2. V1 또는 V2 데이터 자동 선택
# ------------------------------------------------------------

if best_label.startswith("V1_"):
    X_final_train = X_v1_train
    X_final_test = X_v1_test
    selected_variant = "V1"

elif best_label.startswith("V2_"):
    X_final_train = X_v2_train
    X_final_test = X_v2_test
    selected_variant = "V2"

else:
    raise ValueError(f"알 수 없는 데이터 버전입니다: {best_label}")


# ------------------------------------------------------------
# 3. 모델 종류 자동 선택
# ------------------------------------------------------------


In [ ]:
if "LogisticRegression" in best_label:

    logistic_params = {
        key.replace("model__", ""): value
        for key, value in best_params.items()
    }

    final_model = Pipeline(
        steps=[
            (
                "scaler",
                StandardScaler(with_mean=False),
            ),
            (
                "model",
                LogisticRegression(
                    **logistic_params,
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

elif "RandomForest" in best_label:

    final_model = RandomForestClassifier(
        **best_params,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )


elif "XGBoost" in best_label:

    negative = int((y_train == 0).sum())
    positive = int((y_train == 1).sum())

    scale_pos_weight = negative / positive

    final_model = XGBClassifier(
        **best_params,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )


elif "CatBoost" in best_label:

    final_model = CatBoostClassifier(
        **best_params,
        auto_class_weights="Balanced",
        verbose=False,
        allow_writing_files=False,
        random_seed=RANDOM_STATE,
    )


else:
    raise ValueError(f"알 수 없는 모델입니다: {best_label}")


# ------------------------------------------------------------
# 4. Train OOF 예측으로 임계값 선택
# Test 데이터는 임계값 결정에 사용하지 않음
# ------------------------------------------------------------


In [ ]:
oof_probabilities = cross_val_predict(
    estimator=clone(final_model),
    X=X_final_train,
    y=y_train,
    groups=groups_train,
    cv=group_cv,
    method="predict_proba",
    n_jobs=1,
)[:, 1]


threshold_rows = []

for threshold in np.arange(0.05, 0.951, 0.01):

    oof_predictions = (
        oof_probabilities >= threshold
    ).astype(int)

    threshold_rows.append({
        "threshold": threshold,

        "precision": precision_score(
            y_train,
            oof_predictions,
            zero_division=0,
        ),

        "recall": recall_score(
            y_train,
            oof_predictions,
            zero_division=0,
        ),

        "f1": f1_score(
            y_train,
            oof_predictions,
            zero_division=0,
        ),

        "f0.5": fbeta_score(
            y_train,
            oof_predictions,
            beta=0.5,
            zero_division=0,
        ),
    })


threshold_results = pd.DataFrame(threshold_rows)

# Recall이 너무 낮은 threshold 제외
valid_thresholds = threshold_results.loc[
    threshold_results["recall"] >= 0.30
].copy()


In [ ]:
if valid_thresholds.empty:

    best_threshold = 0.5
    print("조건을 만족하는 임계값이 없어 0.5를 사용합니다.")

else:

    best_threshold_row = (
        valid_thresholds
        .sort_values(
            ["f0.5", "precision", "recall"],
            ascending=False,
        )
        .iloc[0]
    )

    best_threshold = float(
        best_threshold_row["threshold"]
    )

    print("\n선택된 OOF threshold")
    print(best_threshold_row)


# ------------------------------------------------------------
# 5. Train 전체로 최종 모델 학습
# ------------------------------------------------------------

final_model.fit(
    X_final_train,
    y_train,
)


# ------------------------------------------------------------
# 6. Test 확률 및 분류 결과
# ------------------------------------------------------------

test_probabilities = final_model.predict_proba(
    X_final_test
)[:, 1]

test_predictions = (
    test_probabilities >= best_threshold
).astype(int)


# ------------------------------------------------------------
# 7. 최종 Test 성능
# ------------------------------------------------------------

print("\n============================================================")
print(f"✅ 데이터 분할 정보: Train {len(y_train)}건 | Test {len(y_test)}건")
print("============================================================\n")


In [ ]:
final_metrics = {
    "variant": selected_variant,
    "model": best_label,
    "threshold": best_threshold,

    "Accuracy": accuracy_score(
        y_test,
        test_predictions,
    ),

    "Precision": precision_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),

    "Recall": recall_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),

    "F1": f1_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),

    "F0.5": fbeta_score(
        y_test,
        test_predictions,
        beta=0.5,
        zero_division=0,
    ),

    "ROC-AUC": roc_auc_score(
        y_test,
        test_probabilities,
    ),

    "PR-AUC": average_precision_score(
        y_test,
        test_probabilities,
    ),
}

final_metrics_df = pd.DataFrame(
    [final_metrics]
)

print("[최종 Test 성능 지표]")
display(final_metrics_df)

print("\n[Test Confusion Matrix]")
print(
    confusion_matrix(
        y_test,
        test_predictions,
    )
)


In [70]:
print("\n[Test Classification Report]")
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=["일반군", "고위험군"],
        zero_division=0,
    )
)

# ------------------------------------------------------------
# 8. Test 예측 결과와 메타데이터(부정률 보정 내용 포함) 결합하여 확인
# ------------------------------------------------------------

test_results_df = metadata_test.copy()
test_results_df["predict_proba"] = test_probabilities
test_results_df["prediction"] = test_predictions
test_results_df["is_correct"] = (test_results_df["high_risk"] == test_results_df["prediction"]).astype(int)

print("\n[Test 예측 결과 샘플 (부정리뷰 보정 지표 포함)]")
display(
    test_results_df[
        [
            "product_id",
            "피부타입",
            "y_total_reviews",
            "y_negative_rate_raw",
            "y_negative_rate_smoothed",
            "high_risk",
            "predict_proba",
            "prediction",
            "is_correct"
        ]
    ].head(15)
)


선택된 최종 후보: V2_RandomForest
CV PR-AUC: 0.3523984504799521
최적 파라미터: {'max_depth': 6, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 200}

선택된 OOF threshold
threshold    0.470000
precision    0.315789
recall       0.432432
f1           0.365019
f0.5         0.333797
Name: 42, dtype: float64

✅ 데이터 분할 정보: Train 869건 | Test 255건

[최종 Test 성능 지표]


,variant,model,threshold,Accuracy,Precision,Recall,F1,F0.5,ROC-AUC,PR-AUC
0,V2,V2_RandomForest,0.47,0.596078,0.341667,0.630769,0.443243,0.376147,0.623036,0.384063



[Test Confusion Matrix]
[[111  79]
 [ 24  41]]

[Test Classification Report]
              precision    recall  f1-score   support

         일반군       0.82      0.58      0.68       190
        고위험군       0.34      0.63      0.44        65

    accuracy                           0.60       255
   macro avg       0.58      0.61      0.56       255
weighted avg       0.70      0.60      0.62       255


[Test 예측 결과 샘플 (부정리뷰 보정 지표 포함)]


,product_id,피부타입,y_total_reviews,y_negative_rate_raw,y_negative_rate_smoothed,high_risk,predict_proba,prediction,is_correct
41,A000000149064,지성,10,0.000000,0.038643,0,0.398522,0,1
83,A000000161804,건성,26,0.076923,0.068680,0,0.490224,1,0
84,A000000161804,민감성,23,0.043478,0.050216,0,0.509760,1,0
85,A000000161804,복합성,39,0.051282,0.053547,0,0.507038,1,0
86,A000000161804,지성,12,0.083333,0.067478,0,0.522627,1,0
87,A000000161804,트러블성,10,0.100000,0.071977,0,0.518253,1,0
118,A000000172040,건성,36,0.027778,0.038559,0,0.438529,0,1
119,A000000172040,민감성,47,0.170213,0.136706,1,0.447915,0,0
120,A000000172040,복합성,34,0.029412,0.039987,0,0.459231,0,1
121,A000000172040,약건성,16,0.000000,0.032203,0,0.427348,0,1


## Test 예측 결과 저장

최종 예측값과 실제 라벨, 확률 및 제품 정보를 `reports/metrics`에 저장합니다.


In [ ]:
# 모델 테스트 결과 저장
test_results_df.to_csv(REPORTS_DIR / "metrics" / "테스트_예측결과.csv", index=False, encoding="utf-8-sig")
print("테스트 결과 저장 완료: 테스트_예측결과.csv")